In [ ]:
# Prepare the raw GPS table.
import pandas as pd 
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg
import geopandas as gpd 
 
cows_f = pd.read_csv("df.csv")

np.random.seed(0) # Set seed for reproducibility.

In [ ]:
print(cows_f.groupby("Serial").size().mean()) 

In [ ]:
# Assigns variables to an object.
continuous1 = ["HDOP", "Horizontal Accuracy", "#Satellites"]

#Calculates descriptive statistics for numeric variables.
numeric1 = cows_f[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = cows_f[column].median()
print(numeric1)

In [ ]:
c400 = cows_f.loc[cows_f['Serial'] == 295400]
c400

In [ ]:
# Converts all values over 200 to a float nan (>200 was used to select outliers after inspecting the 'age' histogram and .unique() outputs).
# Renames columns to remove spaces and hashtags.
# Creates a new copy of the specified columns with a different name.
cows_f["Pasture_ID"] = cows_f["Pasture ID"]
cows_f["Pasture_Version"] = cows_f["Pasture Version"]
cows_f["Message_Type"] = cows_f["Message Type"]
cows_f["Battery_Voltage"] = cows_f["Battery Voltage"]
cows_f["GSM_RAT"] = cows_f["GSM RAT"]
cows_f["GSM_Operator"] = cows_f["GSM Operator"]
cows_f["Position_Time"] = cows_f["Position Time"]
cows_f["Horizontal_Accuracy"] = cows_f["Horizontal Accuracy"]
cows_f["n_Satellites"] = cows_f["#Satellites"]
cows_f["Fence_Status"] = cows_f["Fence Status"]
cows_f["Collar_Status"] = cows_f["Collar Status"]

# Removes the originally named column so only the renamed column remains.
cows_f = cows_f.drop('Pasture ID', axis=1)
cows_f = cows_f.drop('Pasture Version', axis=1)
cows_f = cows_f.drop('Message Type', axis=1)
cows_f = cows_f.drop('Battery Voltage', axis=1)
cows_f = cows_f.drop('GSM RAT', axis=1)
cows_f = cows_f.drop('GSM Operator', axis=1)
cows_f = cows_f.drop('Position Time', axis=1)
cows_f = cows_f.drop('Horizontal Accuracy', axis=1)
cows_f = cows_f.drop('#Satellites', axis=1)
cows_f = cows_f.drop('Fence Status', axis=1)
cows_f = cows_f.drop('Collar Status', axis=1)

# Converts column to datetime.
cows_f["Position_Time"] = pd.to_datetime(cows_f["Position_Time"]) 
#cows_f['Position_Time'] = (cows_f['Position_Time'] - pd.Timestamp('2024-02-09 00:31:27+00')) / pd.Timedelta(days=1)

cows_f.loc[cows_f['Latitude'] > 52.645, 'Latitude'] = np.nan
cows_f.loc[cows_f['Latitude'] < 52.620, 'Latitude'] = np.nan
cows_f.loc[cows_f['Longitude'] > 1.28, 'Longitude'] = np.nan
cows_f.loc[cows_f['Longitude'] < 1.24, 'Longitude'] = np.nan
cows_f.loc[cows_f['Horizontal_Accuracy'] > 150, 'Horizontal_Accuracy'] = np.nan
cows_f.loc[cows_f['Collar_Status'] == "OffAnimal", 'Collar_Status'] = np.nan
cows_f.loc[cows_f['HDOP'] > 200, 'HDOP'] = np.nan
cows_f.loc[cows_f['n_Satellites'] < 5, 'n_Satellites'] = np.nan

# Removes columns.
cows_f = cows_f.drop('Activity', axis=1)
cows_f = cows_f.drop('Mode', axis=1)
cows_f = cows_f.drop('Battery_Voltage', axis=1)
cows_f = cows_f.drop('GSM_RAT', axis=1)
cows_f = cows_f.drop('GSM_Operator', axis=1)
#cows_f = cows_f.drop('Time', axis=1)
#cows_f = cows_f.drop('Collar_Status', axis=1)
#cows_f = cows_f.drop('Fence_Status', axis=1)

# Removes the remaining rows with missing values for the respective fields.
cows_f.dropna(inplace=True)

# Converts data type from float to integer (discrete).
cows_f["Pasture_ID"]=cows_f["Pasture_ID"].astype(int)
cows_f["Pasture_Version"]=cows_f["Pasture_Version"].astype(int)
cows_f["n_Satellites"]=cows_f["n_Satellites"].astype(int)

# data: everything but remove low quality days (less than 1 segment?) 

# HMM quality (determine segments): 
# consecutive points 10-20 mins apart, allowing for the NoFence, must have >4 points (1 hour), filter out all non-segment points

# daily routine: must have >8 hours with >24 records, no gap >2horus

#Compare stats on the above 3 sets 
#Run the HMM (100 seeds) on all data and HMM quality and compare outputs  
#If there’s time, look at the routine, but not central to this part of the work

In [ ]:
# Assigns variables to an object.
continuous1 = ["HDOP", "Horizontal_Accuracy", "n_Satellites"]

#Calculates descriptive statistics for numeric variables.
numeric1 = cows_f[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = cows_f[column].median()
print(numeric1)

In [ ]:
# Convert latitude/longitude to projected coordinates.
gdf = gpd.GeoDataFrame( 
    cows_f, 
    geometry=gpd.points_from_xy(cows_f["Longitude"], cows_f["Latitude"]), 
    crs="EPSG:4326"
) 
 
# Example only: British National Grid. Change if needed. 
gdf = gdf.to_crs("EPSG:27700") 
 
gdf["x"] = gdf.geometry.x 
gdf["y"] = gdf.geometry.y 

In [ ]:
# Calculate movement features.
def add_movement_features(cows_f): 
    cows_f = cows_f.sort_values(["Serial", "Position_Time"]).copy() 
 
    cows_f["x_prev"] = cows_f.groupby("Serial")["x"].shift(1) 
    cows_f["y_prev"] = cows_f.groupby("Serial")["y"].shift(1) 
    cows_f["t_prev"] = cows_f.groupby("Serial")["Position_Time"].shift(1) 
 
    dx = cows_f["x"] - cows_f["x_prev"] 
    dy = cows_f["y"] - cows_f["y_prev"] 
 
    cows_f["step_length_m"] = np.sqrt(dx**2 + dy**2)
    cows_f["time_delta_min"] = (cows_f["Position_Time"] - cows_f["t_prev"]).dt.total_seconds() / 60 
    cows_f["speed_m_per_min"] = cows_f["step_length_m"] / cows_f["time_delta_min"] 
 
    cows_f["heading"] = np.arctan2(dy, dx) 
    cows_f["heading_prev"] = cows_f.groupby("Serial")["heading"].shift(1) 
    turn = cows_f["heading"] - cows_f["heading_prev"] 
    cows_f["turning_angle"] = np.arctan2(np.sin(turn), np.cos(turn))
    cows_f["abs_turning_angle"] = np.abs(cows_f["turning_angle"]) 
 
    cows_f["hour"] = cows_f["Position_Time"].dt.hour 
    cows_f["date"] = cows_f["Position_Time"].dt.date 
    cows_f["month"] = cows_f["Position_Time"].dt.month 
    cows_f["day_of_year"] = cows_f["Position_Time"].dt.dayofyear 
 
    return cows_f
 
gdf = add_movement_features(gdf) 

gdf.to_csv("cow_movement_features.csv", index=False) 

In [ ]:
# Ensure data is datetime format and strictly sorted by cow and time
df = gdf.sort_values(by=["Serial", "Position_Time"]).reset_index(drop=True)

# Calculate the time gap to the PREVIOUS record within the same cow
df["gap_seconds"] = df.groupby("Serial")["Position_Time"].diff().dt.total_seconds()

# Define the 15-minute target window (+/- 30 seconds -> 870 to 930 seconds)
df["is_15min_target"] = df["gap_seconds"].between(870, 930)

# --- NEW: Define the wider 10-20 min window (600 to 1200 seconds) ---
df["is_10_20min_gap"] = df["gap_seconds"].between(600, 1200)

# --- NEW: Calculate consecutive run IDs for the 10-20 min window per Cow ---
# A run ID changes whenever the True/False state changes compared to the previous row
df["run_id"] = (df["is_10_20min_gap"] != df.groupby("Serial")["is_10_20min_gap"].shift()).cumsum()


# Categorise records into parts of the day
def get_day_part(hour):
    if 5 <= hour < 12: return "Morning"
    elif 12 <= hour < 17: return "Afternoon"
    elif 17 <= hour < 21: return "Evening"
    else: return "Night"

df["day_part"] = df["Position_Time"].dt.hour.apply(get_day_part)


# Aggregate by Cow and Day
def analyze_day(group):
    # Basic counts and times
    num_records = len(group)
    first_time = group["Position_Time"].min()
    last_time = group["Position_Time"].max()
    
    # Spans and Gaps
    day_span_hours = (last_time - first_time).total_seconds() / 3600.0
    largest_gap_min = group["gap_seconds"].max() / 60.0 if num_records > 1 else 0.0
    
    # Rates
    records_per_hour = num_records / day_span_hours if day_span_hours > 0 else num_records
    
    # Target 15-min intervals (+/- 30s)
    num_15min_records = group["is_15min_target"].sum()
    
    # Expected records: If active for X hours, expected = (X * 4 intervals/hour)
    expected_records = (day_span_hours * 4) + 1
    pct_expected_captured = (num_15min_records / expected_records) * 100 if expected_records > 1 else 0.0
    
    # Records by day part (returns a string breakdown dictionary)
    day_part_counts = group["day_part"].value_counts().to_dict()
    
    # --- NEW: Calculate Max Consecutive Points 10-20 mins apart ---
    valid_runs = group[group["is_10_20min_gap"]]
    if not valid_runs.empty:
        # Group by the pre-calculated run_id, count intervals, and find the maximum streak
        # Adding 1 converts the number of "gaps" into the number of physical "GPS points"
        max_consec_points = int(valid_runs.groupby("run_id").size().max() + 1)
    else:
        max_consec_points = 0
    
    return pd.Series({
        "n_records": num_records,
        "number_15_minute_(+/-_30_seconds)": int(num_15min_records),
        "max_consecutive_10-20_mins_ap": max_consec_points, # <-- NEW VARIABLE
        "first_record": first_time.strftime("%H:%M:%S"),
        "last_record": last_time.strftime("%H:%M:%S"),
        "day_span_(hours)": round(day_span_hours, 2),
        "largest_gap_minutes": round(largest_gap_min, 2) if not pd.isna(largest_gap_min) else 0.0,
        "records_per_hour": round(records_per_hour, 2),
        "records_by_day_part": str(day_part_counts),
        "%_expected_15_minute_(+/-_30_seconds)_records_captured": round(pct_expected_captured, 1) 
    })

# Run the aggregation
summary_table1 = df.groupby(["Serial", "date"], as_index=False).apply(analyze_day)

summary_table1.to_csv("all_data_summary_table.csv", index=False)

summary_table1

In [ ]:
# 1. Filter the summary table for your condition
valid_groups = summary_table1[summary_table1["max_consecutive_10-20_mins_ap"] >= 0]

# 2. Extract the Serial and date columns that match this condition
valid_keys = valid_groups[["Serial", "date"]]

# 3. Filter df by matching its Serial and date with the valid keys
# (First ensure df has the 'date' column extracted from Position_Time)
df["date"] = df["Position_Time"].dt.date
filtered_df = df.merge(valid_keys, on=["Serial", "date"], how="inner")

filtered_df

In [ ]:
# 1. Filter the summary table for your condition
valid_groups = summary_table1[summary_table1["largest_gap_minutes"] < 120]

# 2. Extract the Serial and date columns that match this condition
valid_keys = valid_groups[["Serial", "date"]]

# 3. Filter df by matching its Serial and date with the valid keys
# (First ensure df has the 'date' column extracted from Position_Time)
filtered_df["date"] = filtered_df["Position_Time"].dt.date
filtered_df_dailyc = filtered_df.merge(valid_keys, on=["Serial", "date"], how="inner")

filtered_df_dailyc

In [ ]:
# Remove rows where the movement calculation is not meaningful, such as the first row for each animal or very large gaps between fixes. 

clean = filtered_df.dropna( 
    subset=["step_length_m", "time_delta_min", "speed_m_per_min", "turning_angle"] 
).copy() 

# Keep roughly 10-minute fixes, allowing some tolerance. 
clean = clean[(clean["time_delta_min"] >= 14.5) & (clean["time_delta_min"] <= 15.5)] 

# Remove implausibly high speeds after inspecting the distribution. 
clean = clean[clean["speed_m_per_min"] < 650] 
#clean = clean[clean["Fence_Status"] == "MaybeOutOfFence"] 

In [ ]:
# Assigns variables to an object.
continuous1 = ["speed_m_per_min", "abs_turning_angle", "step_length_m", "time_delta_min", "HDOP", "Horizontal_Accuracy", "n_Satellites"]

#Calculates descriptive statistics for numeric variables.
numeric1 = clean[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = clean[column].median()
print(numeric1)

In [ ]:
# Ensure data is datetime format and strictly sorted by cow and time
df = clean.sort_values(by=["Serial", "Position_Time"]).reset_index(drop=True)

# Calculate the time gap to the PREVIOUS record within the same cow
df["gap_seconds"] = df.groupby("Serial")["Position_Time"].diff().dt.total_seconds()

# Define the 15-minute target window (+/- 30 seconds -> 870 to 930 seconds)
df["is_15min_target"] = df["gap_seconds"].between(870, 930)

# --- NEW: Define the wider 10-20 min window (600 to 1200 seconds) ---
df["is_10_20min_gap"] = df["gap_seconds"].between(600, 1200)

# --- NEW: Calculate consecutive run IDs for the 10-20 min window per Cow ---
# A run ID changes whenever the True/False state changes compared to the previous row
df["run_id"] = (df["is_10_20min_gap"] != df.groupby("Serial")["is_10_20min_gap"].shift()).cumsum()


# Categorise records into parts of the day
def get_day_part(hour):
    if 5 <= hour < 12: return "Morning"
    elif 12 <= hour < 17: return "Afternoon"
    elif 17 <= hour < 21: return "Evening"
    else: return "Night"

df["day_part"] = df["Position_Time"].dt.hour.apply(get_day_part)


# Aggregate by Cow and Day
def analyze_day(group):
    # Basic counts and times
    num_records = len(group)
    first_time = group["Position_Time"].min()
    last_time = group["Position_Time"].max()
    
    # Spans and Gaps
    day_span_hours = (last_time - first_time).total_seconds() / 3600.0
    largest_gap_min = group["gap_seconds"].max() / 60.0 if num_records > 1 else 0.0
    
    # Rates
    records_per_hour = num_records / day_span_hours if day_span_hours > 0 else num_records
    
    # Target 15-min intervals (+/- 30s)
    num_15min_records = group["is_15min_target"].sum()
    
    # Expected records: If active for X hours, expected = (X * 4 intervals/hour)
    expected_records = (day_span_hours * 4) + 1
    pct_expected_captured = (num_15min_records / expected_records) * 100 if expected_records > 1 else 0.0
    
    # Records by day part (returns a string breakdown dictionary)
    day_part_counts = group["day_part"].value_counts().to_dict()
    
    # --- NEW: Calculate Max Consecutive Points 10-20 mins apart ---
    valid_runs = group[group["is_10_20min_gap"]]
    if not valid_runs.empty:
        # Group by the pre-calculated run_id, count intervals, and find the maximum streak
        # Adding 1 converts the number of "gaps" into the number of physical "GPS points"
        max_consec_points = int(valid_runs.groupby("run_id").size().max() + 1)
    else:
        max_consec_points = 0
    
    return pd.Series({
        "n_records": num_records,
        "number_15_minute_(+/-_30_seconds)": int(num_15min_records),
        "max_consecutive_10-20_mins_ap": max_consec_points, # <-- NEW VARIABLE
        "first_record": first_time.strftime("%H:%M:%S"),
        "last_record": last_time.strftime("%H:%M:%S"),
        "day_span_(hours)": round(day_span_hours, 2),
        "largest_gap_minutes": round(largest_gap_min, 2) if not pd.isna(largest_gap_min) else 0.0,
        "records_per_hour": round(records_per_hour, 2),
        "records_by_day_part": str(day_part_counts),
        "%_expected_15_minute_(+/-_30_seconds)_records_captured": round(pct_expected_captured, 1) 
    })

# Run the aggregation
summary_table2 = df.groupby(["Serial", "date"], as_index=False).apply(analyze_day)

summary_table2.to_csv("15_min+-30_summary_table.csv", index=False)

summary_table2

# A standard 24-hour day expects 96 records. 
# However, because cows might only be tracked for part of a day (e.g., a collar was deployed at noon), 
# the script calculates expected intervals based dynamically on their operational day_span_hours (hours × 4).

In [ ]:
other_features = ["speed_m_per_min", "turning_angle"]

plt.figure(figsize=(12, 10))

for i, feature in enumerate(other_features):
    plt.subplot(2, 2, i + 1)
    plt.scatter("time_delta_min", feature, s=1, c="black", alpha=0.5, data=clean) # Simplified mpl.pyplot.scatter
    plt.ylabel(feature)
    plt.xlabel("time_delta_min")

plt.suptitle('Figure 5. Scatterplots of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("move_feat_by_time_delta(Position_Time_15_mins+-30).png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
other_features = ["speed_m_per_min"]

plt.figure(figsize=(12, 10))

for i, feature in enumerate(other_features):
    plt.subplot(2, 2, i + 1)
    plt.scatter("turning_angle", feature, s=1, c="black", alpha=0.5, data=clean) # Simplified mpl.pyplot.scatter
    plt.ylabel(feature)
    plt.xlabel("turning_angle")

plt.suptitle('Figure 5. Scatterplots of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("velocity_by_angle).png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
other_features = ["speed_m_per_min", "turning_angle"]

plt.figure(figsize=(12, 10))

for i, col in enumerate(other_features):
    plt.subplot(2, 2, i + 1)
    sns.kdeplot(data=clean, x=col, fill=True, common_norm=False, alpha=.5, linewidth=0)
    plt.xlabel(col)
    plt.ylabel("frequency")
plt.suptitle('Figure 2. KDE plots of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("move_feat_KDE.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
c1 = clean.loc[clean['Serial'] == 295830]

plt.figure(figsize=(5, 4))
sns.kdeplot(data=c1, x="time_delta_min", fill=True, common_norm=False, alpha=.5, linewidth=0)
plt.xlabel("time_delta_min")
plt.ylabel("frequency")
plt.suptitle('Figure 2. Histograms of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("time_delta_KDE(Position_Time_15_mins+-30).png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Assigns variables to an object.
continuous1 = ["speed_m_per_min", "abs_turning_angle", "step_length_m", "time_delta_min", "HDOP", "Horizontal_Accuracy", "n_Satellites"]

#Calculates descriptive statistics for numeric variables.
numeric1 = clean[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = clean[column].median()
print(numeric1)

In [ ]:
print((clean).groupby("Serial").size().mean()) 

In [ ]:
print((clean).groupby("Serial").size())

In [ ]:
daily1 = ( 
    clean.groupby(["Serial", "date"]) 
    .size()
    .reset_index(name="frequency") 
) 

daily1

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x='Serial', y="frequency", data=daily1)
plt.title("Frequency of recordings per day")

plt.savefig("individual_recording_frequency(15+-30).png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Then summarise by day, hour, and season.
hourly1 = ( 
    clean.groupby(["Serial", "hour"]) 
    .size()
    .reset_index(name="frequency") 
) 

d1 = hourly1.loc[hourly1['Serial'] == 294366]
d2 = hourly1.loc[hourly1['Serial'] == 295400]
d3 = hourly1.loc[hourly1['Serial'] == 295402]
d4 = hourly1.loc[hourly1['Serial'] == 295770]
d5 = hourly1.loc[hourly1['Serial'] == 295815]
d6 = hourly1.loc[hourly1['Serial'] == 295830]

plt.figure(figsize=(5, 4))
sns.barplot(x='hour', y="frequency", data=hourly1)
plt.title("Frequency of recordings per hour")

plt.savefig("hourly_recording frequency(15+-30).png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Explore before modelling.
plt.figure(figsize=(11, 5)) 
plt.hist(clean["speed_m_per_min"], bins=100) 
plt.xlabel("Speed (m/min)") 
plt.ylabel("Number of movement steps") 
plt.yscale("log") 
plt.tight_layout() 
plt.savefig("cow_speed_distribution.png", dpi=300) 

In [ ]:
# Prepare features.
from sklearn.preprocessing import StandardScaler 
 
hmm_df = clean.copy() 
hmm_df["log_step"] = np.log1p(hmm_df["step_length_m"])
hmm_df["log_speed"] = np.log1p(hmm_df["speed_m_per_min"])
hmm_df["abs_turning_angle"] = np.abs(hmm_df["turning_angle"])

hmm_df = hmm_df.sort_values(["Serial", "Position_Time"]).copy()

hmm_df.to_csv("hmm_df.csv", index=False)

In [ ]:
hmm_df = pd.read_csv("hmm_df_b_R.csv")

In [ ]:
print((daily1["frequency"]).mean()) 

In [ ]:
d1 = hmm_df.loc[hmm_df['Serial'] == 294366]
d2 = hmm_df.loc[hmm_df['Serial'] == 295400]
d3 = hmm_df.loc[hmm_df['Serial'] == 295402]
d4 = hmm_df.loc[hmm_df['Serial'] == 295770]
d5 = hmm_df.loc[hmm_df['Serial'] == 295815]
d6 = hmm_df.loc[hmm_df['Serial'] == 295830]

print((hmm_df["time_delta_min"]).sum()) 
print((d1["time_delta_min"]).sum()) 
print((d2["time_delta_min"]).sum()) 
print((d3["time_delta_min"]).sum()) 
print((d4["time_delta_min"]).sum()) 
print((d5["time_delta_min"]).sum()) 
print((d6["time_delta_min"]).sum()) 

In [ ]:
1140815.8666666667/6

In [ ]:
190135.9777777778 / 1440

In [ ]:
0.03887345679013 * 1440

In [ ]:
0.977777777787196 * 60

In [ ]:
76095 / 264608 * 100

In [ ]:
print((hmm_df["step_length_m"]).sum()) 
print((d1["step_length_m"]).sum()) 
print((d2["step_length_m"]).sum()) 
print((d3["step_length_m"]).sum()) 
print((d4["step_length_m"]).sum()) 
print((d5["step_length_m"]).sum()) 
print((d6["step_length_m"]).sum()) 

In [ ]:
1140815.8666666667 / 1440, 213378.11666666664 / 1440, 123321.65 / 1440, 191329.8333333333 / 1440, 203619.84999999998 / 1440, 203804.44999999998 / 1440, 205361.96666666667 / 1440,

In [ ]:
3023332.7479155613 / 792.2332407407407, 553825.2169398123 / 148.17924768518517, 310499.8914199425 / 85.64003472222223, 521755.5104656388 / 132.8679398148148, 550346.4085442406 / 141.40267361111108, 597599.8773377696 / 141.53086805555554, 489305.84320815746 / 142.61247685185185

In [ ]:
#!/usr/bin/env python3

import argparse
import pandas as pd
from hmmlearn.hmm import GaussianHMM

hmm_df.to_csv("hmm_df.csv", index=False)

def fit_model(X, n_states, seed, n_iter):
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=n_iter,
        random_state=seed,
        tol=1e-3
    )

    model.fit(X)

    return {
        "n_states": n_states,
        "seed": seed,
        "log_likelihood": model.score(X),
        "AIC": model.aic(X),
        "BIC": model.bic(X),
        "converged": model.monitor_.converged,
        "model": model
    }


def plot_model_selection(best_by_states, output_prefix):
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_theme(style="ticks", context="talk")

    plot_df = best_by_states.melt(
        id_vars="n_states",
        value_vars=["AIC", "BIC", "log_likelihood"],
        var_name="metric",
        value_name="value"
    )

    fig, ax1 = plt.subplots(figsize=(10, 6))

    sns.lineplot(
        data=plot_df[plot_df["metric"].isin(["AIC", "BIC"])],
        x="n_states",
        y="value",
        hue="metric",
        marker="o",
        linewidth=2.5,
        ax=ax1
    )

    ax1.set_xlabel("Number of hidden states")
    ax1.set_ylabel("Information criterion\nlower is better")

    ax2 = ax1.twinx()

    ax1.grid(False)
    ax2.grid(False)

    sns.lineplot(
        data=plot_df[plot_df["metric"] == "log_likelihood"],
        x="n_states",
        y="value",
        marker="o",
        linewidth=2.5,
        color="black",
        label="log_likelihood",
        ax=ax2
    )

    ax2.set_ylabel("Log-likelihood\nhigher is better")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        frameon=False,
        loc="best"
    )

    ax2.get_legend().remove()

    sns.despine(right=False)
    fig.tight_layout()

    fig.savefig(f"{output_prefix}.model_selection.png", dpi=300)
    fig.savefig(f"{output_prefix}.model_selection.pdf")

    plt.close(fig)



def main():
    parser = argparse.ArgumentParser(
        description="Fit Gaussian HMMs across state numbers and random seeds."
    )

    parser.add_argument("--input", required=True, help="Input CSV file")
    parser.add_argument("--min_states", type=int, default=2)
    parser.add_argument("--max_states", type=int, default=6)
    parser.add_argument("--seeds", type=int, default=100)
    parser.add_argument("--n_iter", type=int, default=500)
    parser.add_argument(
        "--columns",
        nargs="+",
        default=["speed_m_min", "turn_angle_deg", "step_length_m"],
        help="Columns to use for HMM fitting"
    )

    args = parser.parse_args([
    "--input", "hmm_df.csv",
    "--min_states", "1",
    "--max_states", "6",
    "--seeds", "100",
    "--n_iter", "500",
    "--columns", "speed_m_per_min", "abs_turning_angle"
    ])

    df = pd.read_csv(args.input)

    X = df[args.columns].dropna()

    all_results = []
    all_models = []

    for n_states in range(args.min_states, args.max_states + 1):
        for seed in range(args.seeds):
            try:
                fit = fit_model(X, n_states, seed, args.n_iter)

                all_models.append(fit)

                all_results.append({
                    "n_states": fit["n_states"],
                    "seed": fit["seed"],
                    "log_likelihood": fit["log_likelihood"],
                    "AIC": fit["AIC"],
                    "BIC": fit["BIC"],
                    "converged": fit["converged"]
                })

            except Exception as e:
                all_results.append({
                    "n_states": n_states,
                    "seed": seed,
                    "log_likelihood": None,
                    "AIC": None,
                    "BIC": None,
                    "converged": False,
                    "error": str(e)
                })

    results = pd.DataFrame(all_results)
    results.to_csv("hmm_model_comparison_all_seeds.csv", index=False)

    best_by_states = (
        results.dropna(subset=["AIC"])
        .sort_values("AIC")
        .groupby("n_states")
        .first()
        .reset_index()
    )

    best_by_states.to_csv("hmm_model_comparison_best_by_state.csv", index=False)

    plot_model_selection(best_by_states, "hmm")

    best_fit = min(
        [m for m in all_models if m["converged"]],
        key=lambda x: x["AIC"]
    )

    best_model = best_fit["model"]
    states = best_model.predict(X)

    df.loc[X.index, "hmm_state"] = states
    df.to_csv("hmm_states_output.csv", index=False)

    print("\nBest models by AIC:")
    print(best_by_states)

if __name__ == "__main__":
    main()

In [ ]:
# Fit the model using each individual as a separate sequence. This is important: the model should not treat the end of one animal’s track as 
# if it were followed by the start of another animal’s track. 
from hmmlearn.hmm import GaussianHMM 
 
X_list = [] 
lengths = [] 
 
for Serial, sub in hmm_df.groupby("Serial"): 
    Xi = sub[["speed_m_per_min", "abs_turning_angle"]].values 
    X_list.append(Xi) 
    lengths.append(len(Xi)) 
 
X_all = np.vstack(X_list) 
scaler = StandardScaler() 
X_all = scaler.fit_transform(X_all) 
 
model = GaussianHMM( 
    n_components=4, 
    covariance_type="full", 
    n_iter=500, 
    random_state=43,
    tol=0.001,
) 

model.fit(X_all, lengths=lengths) 
hmm_df["hmm_state_raw"] = model.predict(X_all, lengths=lengths) 

In [ ]:
# Inspect the raw states before naming them.

state_summary = ( 
    hmm_df.groupby("hmm_state_raw") 
    .agg( 
        mean_step_m=("step_length_m", "mean"), 
        median_step_m=("step_length_m", "median"), 
        mean_speed=("speed_m_per_min", "mean"), 
        mean_abs_turn=("abs_turning_angle", "mean"), 
        n=("hmm_state_raw", "size") 
    )
    .reset_index() 
) 

print(state_summary) 


In [ ]:
# The raw state numbers are arbitrary. Label them only after looking at the state summary. 

# Example only: change this mapping after inspecting state_summary. 
state_labels = { 
    0: "Intensive Foraging",
    1: "Resting",
    2: "Transit",
    3: "Extensive Foraging",
}

hmm_df["behaviour_state_g"] = hmm_df["hmm_state_raw"].map(state_labels) 
hmm_df.to_csv("hmm_df_b_R_g.csv", index=False) 

In [ ]:
sns.kdeplot(data=hmm_df, x="speed_m_per_min", hue="behaviour_state", fill=True, common_norm=False, palette="Set1",
   alpha=.5, linewidth=0)

In [ ]:
sns.kdeplot(data=hmm_df, x="turning_angle", hue="behaviour_state", fill=True, common_norm=False, palette="Set1",
   alpha=.5, linewidth=0)

In [ ]:
# A behavioural profile is a summary of how an individual uses its time. Start with percent time in each state. 

profile = ( 
    hmm_df.groupby(["Serial", "behaviour_state"]) #as_index=False
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="frequency")
) 

profile

#profile.to_csv("cow_individual_behaviour_profiles.csv", index=False) 

In [ ]:
profile = ( 
    hmm_df.groupby(["behaviour_state"]) #as_index=False
    .size() 
    .reset_index(name="frequency")
) 

profile

In [ ]:
plt.figure(figsize=(15, 5))
sns.barplot(x='behaviour_state', y="frequency", data=profile)
plt.show()

In [ ]:
15594 / 76039 * 100, 32407 / 76039 * 100, 22567 / 76039 * 100, 5471 / 76039 * 100,

In [ ]:
# A behavioural profile is a summary of how an individual uses its time.

# Pasture_Version: 2 - 75
# Pasture_ID: 55516 NE, 55540 W, 57462 ENW, 79442 SE
# Serial: 294366, 295400, 295402, 295770, 295815, 295830

profile = ( 
    hmm_df.groupby(["Serial", "behaviour_state"]) #as_index=False
    .size() 
    .reset_index(name="frequency")
)

profile = profile.loc[profile['Serial'] == 294366]


plt.figure(figsize=(15, 5))
sns.barplot(x='behaviour_state', y="frequency", data=profile)
plt.show()

#profile.to_csv("cow_individual_behaviour_profiles.csv", index=False) 

In [ ]:
# Then summarise by day, hour, and season. 

def get_season(month): 
    if month in [12, 1, 2]: 
        return "winter" 
    elif month in [3, 4, 5]: 
        return "spring" 
    elif month in [6, 7, 8]: 
        return "summer" 
    else: 
        return "autumn" 
 
hmm_df["season"] = hmm_df["month"].apply(get_season) 
 
daily = ( 
    hmm_df.groupby(["Serial", "date", "behaviour_state"]) 
    .size()
    .groupby(level=[0, 1]) 
    .apply(lambda x: 100 * x / x.sum())
    #.reset_index(name="percent_time") 
) 

monthly = ( 
    hmm_df.groupby(["Serial", "month", "behaviour_state"]) 
    .size()
    .groupby(level=[0, 1]) 
    .apply(lambda x: 100 * x / x.sum())
    #.reset_index(name="percent_time") 
)

seasonal = ( 
    hmm_df.groupby(["Serial", "season", "behaviour_state"]) 
    .size() 
    .groupby(level=[0, 1]) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="percent_time") 
)

pasture = ( 
    hmm_df.groupby(["Serial", "Pasture_Version", "behaviour_state"]) 
    .size()
    .groupby(level=[0, 1]) 
    .apply(lambda x: 100 * x / x.sum())
    #.reset_index(name="percent_time") 
) 

#daily.head(), monthly.head(), seasonal.head(), pasture.head()

monthly

In [ ]:
monthly1 = ( 
    hmm_df.groupby(["Serial", "month", "behaviour_state"]) #as_index=False
    .size() 
    .reset_index(name="frequency")
)

monthly1a = monthly1.loc[monthly1['Serial'] == 294366]
monthly1b = monthly1a.loc[monthly1['month'] == 1]
monthly2 = monthly1a.loc[monthly1['month'] == 2]
monthly3 = monthly1a.loc[monthly1['month'] == 3]
monthly4 = monthly1a.loc[monthly1['month'] == 4]
monthly5 = monthly1a.loc[monthly1['month'] == 5]
monthly6 = monthly1a.loc[monthly1['month'] == 6]
monthly7 = monthly1a.loc[monthly1['month'] == 7]
monthly8 = monthly1a.loc[monthly1['month'] == 8]
monthly9 = monthly1a.loc[monthly1['month'] == 9]
monthly9 = monthly1a.loc[monthly1['month'] == 10]
monthly11 = monthly1a.loc[monthly1['month'] == 11]
monthly12 = monthly1a.loc[monthly1['month'] == 12]

plt.figure(figsize=(11, 4))
sns.barplot(x='behaviour_state', y="frequency", data=monthly1b)
plt.title("January")
plt.show()

monthly1.head(15)

monthly1.to_csv("cow_monthly_behaviour_profiles.csv", index=False) 

In [ ]:
state_gdf = gpd.GeoDataFrame( 
    hmm_df, 
    geometry=gpd.points_from_xy(hmm_df["x"], hmm_df["y"]), 
    crs="EPSG:27700" 
)

#state_gdf.to_file("behaviour_states_full.gpkg", layer='behaviour_state', driver="GPKG") 

In [ ]:
habitat = gpd.read_file("NVC polygons.shp").to_crs(state_gdf.crs)

# Produces a basic plot to visualise the data.
habitat.plot(column="NVC", 
             legend=True,
             cmap="tab20b",
             figsize=(12, 10))

In [ ]:
inv_quad = gpd.read_file("sweet briar sections.shp").to_crs(state_gdf.crs) 

# Produces a basic plot to visualise the data.
inv_quad.plot(column="section")

In [ ]:
# Assigns variables to an object.
continuous1 = ["speed_m_per_min", "abs_turning_angle", "step_length_m", "time_delta_min"]

In [ ]:
# Calculates descriptive statistics for numeric variables.
numeric1 = hmm_df[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = hmm_df[column].median()
print(numeric1)

In [ ]:
habitat = gpd.read_file("NVC polygons.shp").to_crs(state_gdf.crs) 

# 1. Ensure habitat matches state_gdf's CRS before joining
habitat_aligned = habitat[["NVC", "geometry"]].to_crs(state_gdf.crs)

# 2. Run the spatial join using "intersects" (safest for points-in-polygons)
joined = gpd.sjoin(
    state_gdf, 
    habitat_aligned, 
    how="left", 
    predicate="intersects"
)

joined.replace({"NVC": 'woodland'}, 'Woodland', inplace=True)

In [ ]:
habitat_state = ( 
    joined.groupby(["NVC", "behaviour_state"]) 
    .size() 
    #.groupby(level=0) 
    #.apply(lambda x: 100 * x / x.sum()) 
    .reset_index(name="frequency") 
) 

#joined.to_csv("processed_plus_habitat.csv", index=False) 
habitat_state.to_csv("habitat_by_behaviour_state_full.csv", index=False)

habitat_state

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

# Load your datasets
# Replace with your actual EPSG code for a metric CRS (e.g., EPSG:32631 for UTM 31N)
METRIC_CRS = "EPSG:27700" 

site_boundary = gpd.read_file("NVC polygons.shp").to_crs(METRIC_CRS)
cattle_df = gpd.GeoDataFrame(
    state_gdf, 
    geometry=gpd.points_from_xy(state_gdf['Longitude'], state_gdf['Latitude']), 
    crs="EPSG:27700" # Assuming raw data is in Lat/Lon WGS84
).to_crs(METRIC_CRS)

In [ ]:
# 1. Load and clean the habitat shapefile
METRIC_CRS = "EPSG:4326" 
site_boundary = gpd.read_file("NVC polygons.shp").to_crs(METRIC_CRS)

# --- NEW: CRITICAL TOPOLOGY REPAIR STEP ---
# Check if any geometries are broken
if not site_boundary.geometry.is_valid.all():
    print("Invalid geometries found in shapefile! Repairing...")
    # .make_valid() fixes self-intersections and structural errors automatically
    site_boundary['geometry'] = site_boundary.geometry.make_valid()
# ------------------------------------------

# 2. Generate the Grid
xmin, ymin, xmax, ymax = site_boundary.total_bounds
grid_size = 0.0003
x_coords = np.arange(xmin, xmax, grid_size)
y_coords = np.arange(ymin, ymax, grid_size)

grid_cells = []
for x in x_coords:
    for y in y_coords:
        grid_cells.append(Polygon([
            (x, y), 
            (x + grid_size, y), 
            (x + grid_size, y + grid_size), 
            (x, y + grid_size)
        ]))

grid = gpd.GeoDataFrame({'grid_id': range(len(grid_cells))}, geometry=grid_cells, crs=METRIC_CRS)

# Reproject site_boundary to match the grid CRS
site_boundary = site_boundary.to_crs(grid.crs)

grid.plot(column="grid_id", 
             legend=True,
             cmap="tab20b",
             figsize=(12, 10))

In [ ]:
# Spatial join: Assign a grid_id to each cattle point record
cattle_with_grid = gpd.sjoin(cattle_df, grid, how='left', predicate='within')

# Calculate total records to establish total time proportions
total_records = len(cattle_with_grid)

# Count how many records fall into each grid cell
cell_counts = cattle_with_grid['grid_id'].value_counts()

# Convert counts into proportions (Time Access Ratio)
cell_availability = (cell_counts / total_records).to_dict()

# Map the availability ratio back to the respective grid cells
grid['availability'] = grid['grid_id'].map(cell_availability).fillna(0)

# Map the availability ratio back into each individual point record row
cattle_with_grid['availability'] = cattle_with_grid['grid_id'].map(cell_availability).fillna(0)

cattle_with_grid['availability']

In [ ]:
# 1. Spatial join: Only retain points that successfully hit a 20m grid cell
cattle_mapped = gpd.sjoin(cattle_df, grid[['grid_id', 'geometry']], how='inner', predicate='within')

# 2. Count occurrences per cell relative to valid on-site points
valid_points_total = len(cattle_mapped)
cell_counts = cattle_mapped['grid_id'].value_counts()
cell_availability = (cell_counts / valid_points_total).to_dict()

# 3. Map availability back to the grid and the point dataframes
grid['availability'] = grid['grid_id'].map(cell_availability).fillna(0)

# Merge data back into your master cattle dataframe safely
cattle_df['grid_id'] = gpd.sjoin(cattle_df, grid[['grid_id', 'geometry']], how='left', predicate='within')['grid_id']
cattle_df['availability'] = cattle_df['grid_id'].map(cell_availability).fillna(0)

print(f"Success! Non-zero values generated: {(cattle_df['availability'] > 0).sum()} rows.")

cattle_df['availability']

In [ ]:
# Spatial join: Intersect the 20m grid blocks with the underlying Habitat polygons
grid_with_habitats = gpd.sjoin(grid, site_boundary, how='inner', predicate='intersects')

# Aggregate actual usage (availability) vs geographic area per habitat
habitat_summary = grid_with_habitats.groupby('NVC').agg(
    total_grid_area=('geometry', lambda x: len(x) * 400), # 20x20 = 400m2 per cell
    observed_usage=('availability', 'sum')
).reset_index()

# Calculate proportions
total_area = habitat_summary['total_grid_area'].sum()
habitat_summary['expected_usage_ratio'] = habitat_summary['total_grid_area'] / total_area

# Calculate the Grazing Index: Observed Presence / Expected Baseline Area
# Index > 1 means Over-grazed; Index < 1 means Under-grazed
habitat_summary['grazing_index'] = habitat_summary['observed_usage'] / habitat_summary['expected_usage_ratio']
print(habitat_summary[['NVC', 'grazing_index']])

In [ ]:
# Check the coordinate ranges
print("--- GEOMETRY BOUNDS CHECK ---")
print("Grid Extents:   ", grid.total_bounds)
print("Cattle Extents: ", cattle_df.total_bounds)

# Check if ANY intersection is happening naturally
points_inside = cattle_df.within(grid.unary_union).sum()
print(f"Points physically inside the site boundary: {points_inside} out of {len(cattle_df)}")

In [ ]:
#exactcows = cows.loc[cows['is_location_exact'] == 't']

jf = joined.loc[joined['season'] == "Winter"]

plt.figure(figsize=(12, 10))

sns.scatterplot(
    x='Longitude',
    y='Latitude',
    hue='NVC', # Could try and add stuff like step length or turning angle here.
    style='Serial',
    data=joined,
    palette='coolwarm',
    alpha=0.6,
    size=0.1,
    edgecolor=None)

plt.title('Figure 12. Geospatial Distribution of Cattle Movement Coloured by Month')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Month')
plt.show()

In [ ]:
hab = ( 
    joined.groupby(["NVC", "behaviour_state"]) 
    .size()
    .reset_index(name="frequency") 
) 

h1 = hab.loc[hab['NVC'] == "Bankside veg"]

plt.figure(figsize=(5, 4))
sns.barplot(x='behaviour_state', y="frequency", data=h1)
plt.title("Bankside Vegetation Behavior Distribution")
plt.show()

In [ ]:
# Converts column to datetime (has to be done prior to statistics and visualisations).
#joined['Position_Time'] = pd.to_datetime(cows['Position_Time'])
joined['pt_numeric'] = (joined['Position_Time'] - pd.Timestamp('2024-02-09 00:00:00+00')) / pd.Timedelta(days=1)
joined['pt_numeric']

In [ ]:
print(state_summary) 

base_whole = habitat.plot(color='white', edgecolor='black', figsize=(12, 12))
#base = habitat1.plot(color='white', edgecolor='black', figsize=(12, 12))

#jf = joined.loc[joined['Serial'] == 294366]
#jf = joined.loc[joined['Collar_Status'] == "OffAnimal"]
#jf = joined.loc[joined['Fence_Status'] == "NotStarted"]
jf = joined.loc[joined['hmm_state_raw'] <= 3]
#jf = jf.loc[jf['Serial'] == 294366]
#jf = jf.loc[jf['pt_numeric'] > ]
#jf = jf.loc[jf['pt_numeric'] < 500]

# Plot onto habitat map.
#fig, ax = plt.subplots(figsize=(10, 10))
#joined.plot(column='behaviour_state', 
joined.plot(column='behaviour_state',
                 legend=True, 
                 ax=base_whole, 
                 cmap='tab20b',
                 c="b",
                 alpha=0.6,
                 markersize=0.5)
#all_behavior_states

plt.savefig("all_states.png") 

In [ ]:
#exactcows = cows.loc[cows['is_location_exact'] == 't']

import geopandas as gpd
import movingpandas as mpd
from shapely.geometry import Point
from datetime import datetime

# Pasture_Version: 2 - 75
# Pasture_ID: 55516 NE, 55540 W, 57462 ENW, 79442 SE
# Serial: 294366, 295400, 295402, 295770, 295815, 295830

state_labels = { 
    0: "0",
    1: "1",
    2: "2",
    3: "3",
}

joined["behaviour_state"] = joined["hmm_state_raw"].map(state_labels) 

habitat1 = habitat.loc[habitat['NVC'] == "MG5"]

base = habitat1.plot(color='white', edgecolor='black', figsize=(12, 15))

j_filtered = joined.loc[joined['Serial'] == 295770]
#j_filtered = j_filtered.loc[joined['hmm_state_raw'] == 0]
j_filtered = j_filtered.loc[j_filtered['pt_numeric'] > 392]
j_filtered = j_filtered.loc[j_filtered['pt_numeric'] < 393]

print(j_filtered["Position_Time"].min(), j_filtered["Position_Time"].max()) 

# 1. Assume your data is a GeoPandas DataFrame with point geometries
# Ensure your datetime column is set as the index 
traj_joined = j_filtered.set_index('Position_Time')

# 2. Create the MovingPandas Trajectory 
# Replace 'animal_id' with your actual tracking ID column
traj = mpd.Trajectory(traj_joined, 'Serial', t='Position_Time', crs="EPSG:27700") 

# 3. Plot the trajectory coloured by your behavioral state
#fig, ax = plt.subplots(figsize=(10, 10))
traj.plot(column='behaviour_state', legend=True, ax=base, cmap="coolwarm")
#plt.show()

plt.savefig("traj_example.png") 

In [ ]:
import folium
from folium.plugins import HeatMap

# next steps could include heatmap/raster weighted by grazing per habitat (as a ratio of time spent there possibly), montly or by pasture version, aand filter by fence status.

#exactcows = joined.loc[cows['is_location_exact'] == 't']
jf = joined.loc[joined['hmm_state_raw'] == 1]

geo_data = jf[['Latitude', 'Longitude']] # Can change "cows to vairious filters from last markdown cell.

#geo_data['weight'] = geo_data['hmm_state_raw'] / geo_data['hmm_state_raw'].max()

heat_data = geo_data[['Latitude', 'Longitude']].values.tolist()

map_center = [geo_data['Latitude'].mean(), geo_data['Longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=16)

HeatMap(heat_data, radius=3, blur=1, max_zoom=12).add_to(m)

m